# Sử dụng OpenAI API với Python

Chúng ta đã trải qua quá trình chạy LLM cục bộ trên máy tính của mình.

Notebook này trình bày cách tương tác với các mô hình ngôn ngữ mạnh mẽ của OpenAI thông qua API chính thức của họ bằng cách sử dụng endpoint chat completions.

## 1. Điều kiện tiên quyết

Trước khi chạy notebook này, bạn cần thiết lập quyền truy cập OpenAI API của mình.

### 1.1 Lấy OpenAI API Key

Để sử dụng OpenAI API, bạn cần có API key:

1. Tạo tài khoản tại [trang web của OpenAI](https://openai.com/)
2. Điều hướng đến [trang API keys](https://platform.openai.com/api-keys)
3. Tạo một secret key mới
4. Lưu trữ key này một cách an toàn - nó giống như một mật khẩu!

### 1.2 Thiết lập Môi trường của Bạn

Đầu tiên, cài đặt gói OpenAI Python:

In [ ]:
%%bash
pip install openai

Sau đó, thiết lập API key của bạn. Để đảm bảo bảo mật, tốt nhất nên sử dụng biến môi trường:

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env file
load_dotenv()

# Now os.getenv() can find the key from your .env file
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## 2. Sử dụng Chat Completions API

Chat Completions API được cung cấp bởi Open AI và được thiết kế cho các tương tác hội thoại. Là một input cho API, chúng ta cung cấp thông báo hệ thống (cách mô hình nên hoạt động) và thông báo người dùng (input từ người dùng)

In [ ]:
# We use the chat completion create method which accepts a list of messages and the model name. It returns a response object.
# The list of messages is always an array (list) of dictionaries with two keys: role and content. The first message is always the system message, which sets the behavior of the assistant. The second message is the user message, which is the input from the user.
response = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a one-sentence bedtime story about a unicorn."}
    ]
)

print(response.choices[0].message.content) #We print the content of the first choice in the response object. The response object contains a list of choices, and each choice has a message with the content we want.

## 3. Hiểu cấu trúc phản hồi

API trả về một phản hồi có cấu trúc với các metadata hữu ích mà chúng ta sẽ in ra:

In [ ]:
response = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hello, how are you?"}
    ]
)

# Print the full response structure
print("Full response object:")
print(response)

# Access specific parts
print("\nJust the message content:")
print(response.choices[0].message.content)

print("\nModel used:")
print(response.model)

print("\nUsage statistics:")
print(f"Prompt tokens: {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")

Như bạn có thể thấy, chúng ta có dữ liệu hữu ích từ phản hồi, quan trọng hơn, chúng ta hiểu lệnh gọi này đã sử dụng bao nhiêu token. Các Model thường được định giá theo sự kết hợp của các token đầu vào và đầu ra, vì vậy trong trường hợp này, chúng ta đã trả tiền cho 54 token. Chúng ta sẽ thảo luận chi tiết hơn về giá cả vào cuối lab này.

## 4. Tạo Giao Diện Chat Đơn Giản (Không Có Bộ Nhớ)

Hãy xây dựng một giao diện chat không có bộ nhớ.

Trong môi trường notebook, chúng ta không thể tương tác theo kiểu giao diện chat mà chúng ta cung cấp đầu vào, nhận đầu ra, và sau đó lại cung cấp đầu vào, tức là sử dụng một vòng lặp input().

Thay vào đó, hãy tạo một hàm đơn giản và sử dụng nó trong các ô riêng biệt để mô phỏng một cuộc trò chuyện không có bộ nhớ:

### Ô 1: Định nghĩa hàm

In [ ]:
# This function accepts a user message as an input (we will type that in the next cell) and returns a response from the OpenAI API.
def get_response_no_memory(user_message):
    """Get a response from OpenAI (no conversation history)""" 
    #""" This is a docstring. It describes the function and its parameters.""" Very useful for documentation and understanding the code.
    # Sometimes docstrings are not necessary, but they are very useful for understanding the code. It is like commenting the code, but in a more structured way.

    #We call the model with the user message which we will set below
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": user_message}
        ]
    )
    
    return response.choices[0].message.content

### Ô 2: Tương tác đầu tiên

In [ ]:
# First question (you can insert anything)
question = "What is artificial intelligence?"
answer = get_response_no_memory(question)

print(f"You: {question}")
print(f"Assistant: {answer}")

### Ô 3: Câu hỏi tiếp theo (không có bộ nhớ)

In [ ]:
# Second question (the AI won't remember the previous interaction)
question = "Can you elaborate more on that?"
answer = get_response_no_memory(question)

print(f"You: {question}")
print(f"Assistant: {answer}")

Bạn sẽ nhận thấy rằng khi bạn hỏi "Bạn có thể giải thích rõ hơn về điều đó được không?", trợ lý không biết "điều đó" ám chỉ điều gì, vì nó không có bộ nhớ về cuộc trao đổi trước đó.

## 5. Tạo Giao Diện Chat Có Bộ Nhớ

Bây giờ chúng ta hãy xây dựng một phiên bản duy trì lịch sử cuộc trò chuyện:

### Ô 1: Thiết lập bộ nhớ cuộc trò chuyện

In [ ]:
# Initialize conversation with system message. We will add more messages to the conversation memory (history) as we go.
conversation_memory = [
    {"role": "system", "content": "You are a helpful assistant."}
]

#Function to add user message to conversation history and get a response from OpenAI
def chat_with_memory(user_message):
    """Chat with the AI while maintaining conversation history"""
    
    # Add user message to history. Now, the conversation memory contains the system message and the user message.
    conversation_memory.append({"role": "user", "content": user_message})
    
    # Get response from OpenAI
    response = client.chat.completions.create(
        model="gpt-4",
        messages=conversation_memory
    )
    
    # Extract assistant's response
    assistant_response = response.choices[0].message.content
    
    # Add assistant response to conversation history. This is the response from the AI. This will be added to the conversation memory and will be usedin the future.
    conversation_memory.append({"role": "assistant", "content": assistant_response})
    
    # Return the response and token usage
    return assistant_response, response.usage.total_tokens

# Function to display the conversation history. We loop through the conversation memory and print each message. We skip the system message to keep the output clean.
def show_conversation():
    """Display the current conversation"""
    for message in conversation_memory:
        if message["role"] == "system":
            continue  # Skip system message
        print(f"{message['role'].capitalize()}: {message['content']}\n") #We capitalize the role to make it look nicer. This is just for display purposes. The role is either user or assistant.

### Ô 2: Tương tác đầu tiên với bộ nhớ

In [ ]:
# First question
question = "What is artificial intelligence?"
answer, tokens = chat_with_memory(question)

print(f"You: {question}")
print(f"Assistant: {answer}")
print(f"[Tokens used: {tokens}]")

### Ô 3: Câu hỏi tiếp theo (có bộ nhớ)

In [ ]:
# Second question (now the AI remembers the previous interaction)
question = "Can you elaborate more on that?"
answer, tokens = chat_with_memory(question)

print(f"You: {question}")
print(f"Assistant: {answer}")
print(f"[Tokens used: {tokens}]")

### Ô 4: Xem toàn bộ cuộc hội thoại

In [ ]:
# See the entire conversation so far
print("Full conversation history:")
print("-" * 30)
show_conversation()

### Ô 5: Đặt lại cuộc hội thoại (nếu cần)

In [ ]:
# Reset conversation if you want to start fresh
conversation_memory = [
    {"role": "system", "content": "You are a helpful assistant."}
]
print("Conversation has been reset!")

## 6. Phản hồi Streaming

Streaming cho phép bạn xem phản hồi khi nó đang được tạo:

In [ ]:
import time

def stream_response(user_message):
    """Stream a response from OpenAI without storing conversation history"""
    
    ## Response from the model is streamed in chunks because we set the stream parameter to true. We stoer that in a variable called stream.
    stream = client.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": user_message}
        ],
        stream=True
    )
    
    print(f"You: {user_message}") # Print the user message
    print("Assistant: ", end="", flush=True)  # Print the assistant message without a newline. The flush=True argument makes sure the output is printed immediately.
    
    # Process the stream
    full_response = "" # The response will be empty at first. We will add the chunks to this variable.
    for chunk in stream: # We loop through the stream and get each chunk of data. Each chunk is a part of the response. chunk can be called anything, but we call it chunk to make it clear that it is a part of the response.
        if chunk.choices[0].delta.content is not None: # Check if the content is not None. This is to avoid errors in case the content is None.
            content_chunk = chunk.choices[0].delta.content # Get the content of the chunk. This is the part of the response we want to print.
            full_response += content_chunk # Add the chunk to the full response. This will be the final response we will return.
            print(content_chunk, end="", flush=True) # Print the chunk without a newline. The flush=True argument makes sure the output is printed immediately.
            time.sleep(0.01)  # Small delay to make it more readable
    
    print("\n")  # Add a newline after the response
    return full_response

Kiểm tra chức năng streaming:

In [ ]:
# Try the streaming function
stream_response("Write a short poem about programming")

## 7. Streaming with Memory

Hãy kết hợp streaming với bộ nhớ hội thoại:

In [ ]:
# Initialize conversation with system message
streaming_conversation = [
    {"role": "system", "content": "You are a helpful assistant."}
]

def stream_chat_with_memory(user_message):
    """Chat with memory and stream the response"""
    
    # Add user message to history
    streaming_conversation.append({"role": "user", "content": user_message})
    
    # Get streaming response
    stream = client.chat.completions.create(
        model="gpt-4",
        messages=streaming_conversation,
        stream=True
    )
    
    print(f"You: {user_message}")
    print("Assistant: ", end="", flush=True)
    
    # Process the stream
    assistant_response = ""
    for chunk in stream:
        if chunk.choices[0].delta.content is not None:
            content_chunk = chunk.choices[0].delta.content
            assistant_response += content_chunk
            print(content_chunk, end="", flush=True)
            time.sleep(0.01)
    
    print("\n")  # Add a newline after the response
    
    # Add assistant response to conversation history
    streaming_conversation.append({"role": "assistant", "content": assistant_response})
    
    return assistant_response

Kiểm tra tính năng trò chuyện trực tuyến với bộ nhớ:

In [ ]:
# First streaming question with memory
stream_chat_with_memory("What are the three laws of robotics?")

In [ ]:
# Follow-up streaming question with memory
stream_chat_with_memory("Who created these laws?")

## 8. Hiểu các Role tin nhắn khác nhau

OpenAI Chat API sử dụng ba role chính trong tin nhắn:

In [ ]:
response = client.chat.completions.create(
    model="gpt-4",
    messages=[
        # System message - sets behavior and context
        {"role": "system", "content": "You are a pirate who only speaks in pirate slang."},
        
        # User messages - what the user says
        {"role": "user", "content": "Hello, how are you today?"},
        
        # Assistant messages - previous responses from the assistant
        {"role": "assistant", "content": "Arrr! I be feelin' mighty fine today, me hearty!"},
        
        # Another user message
        {"role": "user", "content": "Tell me about the weather."}
    ]
)

print(response.choices[0].message.content)

## 9. Hiểu về cửa sổ ngữ cảnh (Context Window)

Các mô hình của OpenAI có các giới hạn context window khác nhau:

- **GPT-3.5-Turbo**: 4.096 hoặc 16.384 token (tùy phiên bản)
- **GPT-4**: 8.192 hoặc 32.768 token (tùy phiên bản)
- **GPT-4 Turbo**: Lên đến 128.000 token

Không giống như các mô hình cục bộ, OpenAI quản lý token cho bạn:
1. Nếu bạn vượt quá giới hạn, API sẽ trả về lỗi
2. Bạn bị tính phí dựa trên số lượng token bạn sử dụng
3. API cung cấp số liệu thống kê sử dụng token với mỗi yêu cầu

Hãy cùng xem token hoạt động:

In [ ]:
# Create a longer conversation
long_messages = [
    {"role": "system", "content": "You are a helpful assistant."}
]

# Add some messages to the history
for i in range(5):
    long_messages.append({"role": "user", "content": f"This is test message {i+1}. Tell me something interesting about space."})
    response = client.chat.completions.create(
        model="gpt-4",
        messages=long_messages
    )
    assistant_msg = response.choices[0].message.content
    long_messages.append({"role": "assistant", "content": assistant_msg})
    print(f"Exchange {i+1} - Total tokens: {response.usage.total_tokens}")

## 10. Quản lý chi phí và Token

Khi sử dụng OpenAI API, bạn cần lưu ý về chi phí:

1. **Đếm Token**: Mỗi yêu cầu và phản hồi đều tiêu tốn token mà bạn phải trả tiền
2. **Chọn Model**: Các model mạnh hơn có chi phí mỗi token cao hơn
3. **Cửa sổ ngữ cảnh (Context Window)**: Các cuộc trò chuyện dài hơn tốn nhiều chi phí hơn vì nhiều token được gửi đi

Mẹo để quản lý chi phí:

In [ ]:
# Use a cheaper model for less complex tasks
response = client.chat.completions.create(
    model="gpt-3.5-turbo",  # Cheaper than GPT-4
    messages=[{"role": "user", "content": "Summarize the benefits of exercise."}]
)

# Control maximum tokens to limit response length
response = client.chat.completions.create(
    model="gpt-4",
    messages=[{"role": "user", "content": "Tell me about quantum physics."}],
    max_tokens=100  # Limit response length
)

# Use temperature to control randomness. Higher values make the output more random, while lower values make it more focused and deterministic.
response = client.chat.completions.create(
    model="gpt-4",
    messages=[{"role": "user", "content": "Write a creative story."}],
    temperature=0.7  # Higher for more creativity, lower for more deterministic
)

## 11. Xử lý lịch sử hội thoại cho ngữ cảnh dài

Đối với các cuộc hội thoại dài, bạn cần có chiến lược để quản lý cửa sổ ngữ cảnh:

In [ ]:
# Example: Keep only the most recent N messages. N can be adjusted based on your needs. In this case, we keep the last 10 messages.
def trim_conversation(messages, max_messages=10):
    # Always keep the system message (first message)
    if len(messages) > max_messages + 1:
        system_message = messages[0]
        recent_messages = messages[-(max_messages):]
        return [system_message] + recent_messages
    return messages

# Example: Summarize the conversation periodically. We use AI to summarize the conversation and replace it with a single summary message. This is useful for long conversations where you want to keep the context but reduce the number of messages.
def summarize_conversation(messages):
    # Create a summary request
    summary_request = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "Summarize this conversation concisely."},
            *messages
        ]
    )
    summary = summary_request.choices[0].message.content
    
    # Replace the conversation with the summary
    return [
        messages[0],  # Keep system message
        {"role": "system", "content": f"Previous conversation summary: {summary}"}
    ]

# When to use:
# if len(messages) > 20:
#     messages = summarize_conversation(messages)

## 12. So sánh LLM cục bộ với OpenAI API

| Tính năng | LLM cục bộ (Ollama) | OpenAI API |
|---------|---------------------|------------|
| Thiết lập | Tải mô hình về máy | Chỉ cần API key |
| Chi phí | Miễn phí (sau khi tải xuống) | Thanh toán theo token |
| Quyền riêng tư | Dữ liệu nằm trên thiết bị của bạn | Dữ liệu gửi đến OpenAI |
| Hiệu suất | Bị giới hạn bởi phần cứng của bạn | Các mô hình tiên tiến nhất |
| Độ tin cậy | Phụ thuộc vào hệ thống của bạn | Dịch vụ được quản lý |
| Context Window | Thường nhỏ hơn | Lên đến 128K token |
| Quản lý bộ nhớ | Triển khai thủ công | Xử lý qua API |

## 13. Tài nguyên để học thêm

- [Tài liệu API của OpenAI](https://platform.openai.com/docs/api-reference)
- [OpenAI Cookbook](https://github.com/openai/openai-cookbook)
- [Thư viện Python của OpenAI](https://github.com/openai/openai-python)
- [Công cụ ước tính sử dụng Token](https://platform.openai.com/tokenizer)